# 3. Backends and execution options

Every backend consumes the **same canonical static plan**: the tree, the
interaction lists, the operator mathematics and the requested precision never
depend on where the operators are applied. Backend and execution options only
change *how* the plan is executed. This tutorial measures those choices on one
small problem and says which of them most users should leave on `AUTO`.

| Option | Values | Leave on default? |
|---|---|---|
| `backend` | `AUTO`→`CPU_STATIC`, `CPU_REFERENCE`, `CPU_STATIC`, `CUDA_PARTIAL`, `CUDA_FULL` | choose the device explicitly; `AUTO` never selects CUDA |
| `static_matrix_backend` | `PORTABLE`, `ONE_MKL` | yes; oneMKL accelerates M2L only and is explicit |
| `precision` | `FLOAT32` (default), `FLOAT64` | choose by accuracy need; FP32 halves memory and traffic |
| `expansion_basis` | `SPHERICAL` (default), `CARTESIAN` | yes; spherical stores $(p+1)^2$ instead of $(p+1)(p+2)(p+3)/6$ coefficients |
| `spatial_layout` | `GENERAL`, `REGULAR_GRID` | set `REGULAR_GRID` for lattices of identical bodies; it is only a hint |
| `p2p_packing` | `AUTO`, `CANONICAL_AOS`, `PARTICLE_ROW_SOA`, `TENSOR_DICTIONARY`, `LEAF_BLOCK`, `CUDA_BSR3`, `POINT_GEOMETRY` | yes; forcing a packing is for benchmarks and validation |
| `point_expansion_execution` | `AUTO`, `PRECOMPUTED`, `PROCEDURAL` | yes; the measured policy per backend and precision |
| `use_reduced_symmetry_p2p`, `cuda_dictionary_target_owned`, `cuda_dictionary_power2_microtiles` | booleans | yes; explicit dictionary executor selection |
| `cuda_p2p_bsr_max_bytes` | bytes | retained for compatibility; no longer steers the automatic policy |

The result is identical for every backend and packing up to rounding; an
option a plan cannot honour raises `ValueError` at construction with the
reason.

In [ ]:
import time

import numpy as np

import cdfmm
from tutorial_utils import (
    field_error_summary, lattice_positions, print_table, quiet_construction,
)

positions = lattice_positions(8, 3.0e-8)               # 512 points, 30 nm pitch
particle = np.arange(len(positions), dtype=float)
moments = 1.0e-21 * np.column_stack((
    0.7 + 0.2 * np.sin(0.17 * particle),
    -0.3 + 0.2 * np.cos(0.11 * particle),
    0.4 * np.sin(0.07 * particle + 0.3),
))
identities = np.arange(len(positions), dtype=np.int32)

reference = cdfmm.DenseDirectPlan(
    positions, positions, target_source_indices=identities.tolist(),
    static_precision="float64",
).evaluate(moments)


def base_options(order=4, depth=3, precision=cdfmm.StaticPrecision.FLOAT32,
                 backend=cdfmm.ExecutionBackend.CPU_STATIC):
    options = cdfmm.UniformFmmOptions()
    options.expansion_order = order
    options.tree.max_level = depth
    options.precision = precision
    options.backend = backend
    options.fixed_target_source_indices = identities.tolist()
    return options


def measure(label, options, repeats=5):
    start = time.perf_counter()
    with quiet_construction():
        plan = cdfmm.UniformFmm(positions, positions, options)
    setup = time.perf_counter() - start
    plan.evaluate(moments, target_source_indices=identities)          # warm up
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        field = plan.evaluate(moments, target_source_indices=identities)["H"]
        samples.append(time.perf_counter() - start)
    statistics = plan.static_plan_statistics
    return {
        "case": label,
        "precision": str(field.dtype),
        "relative L2": field_error_summary(field, reference)["relative_l2"],
        "setup s": setup,
        "eval ms": 1e3 * float(np.median(samples)),
        "host MiB": statistics["total_persistent_bytes"] / 2**20,
        "p2p packing": str(plan.p2p_execution_packing).rsplit(".", 1)[-1],
        "p2m": str(plan.p2m_execution).rsplit(".", 1)[-1],
    }, plan


print(f"CUDA: {cdfmm.cuda_full_available()}; oneMKL: {cdfmm.one_mkl_available()}")

## Precision

`FLOAT32` is a true single-precision plan: operators, coefficient state,
scratch, transfers and device buffers are all 32-bit; positions and the
analytical operator construction stay in double precision and each finished
operator is quantised once. Internally an FP32 plan also scales coordinates by
the root width so that nanometre geometry stays representable.

At depth 3 the error below is the far-field truncation error, which is the
same in both precisions; the precision floor becomes visible when the
truncation error is removed. A depth-1 tree has no far field at all (every
leaf is a near neighbour of every other), so its rows show the floor of the
exact near field: a few $10^{-8}$ relative for FP32, and about $10^{-9}$ for
FP64 rather than machine precision, because a plan canonicalises the
normalised positions to a $10^{-9}$ grid of the root box so that translated or
rescaled copies of one geometry share a cache key (tutorial 4).

In [ ]:
rows = []
for depth, orders in ((3, (2, 4, 6)), (1, (4,))):
    for order in orders:
        for precision in (cdfmm.StaticPrecision.FLOAT32, cdfmm.StaticPrecision.FLOAT64):
            row, _ = measure(f"CPU_STATIC p={order} depth={depth}",
                             base_options(order=order, depth=depth, precision=precision))
            rows.append(row)
print_table(rows, ["case", "precision", "relative L2", "eval ms", "host MiB"],
            {"relative L2": ".2e", "eval ms": ".3f", "host MiB": ".2f"})

## Backends

| Backend | Where the work runs | Notes |
|---|---|---|
| `CPU_REFERENCE` | dynamic Cartesian traversal on the CPU | validation and teaching; Cartesian basis only, point geometry only |
| `CPU_STATIC` | portable CPU with prepared operators | the production default and the `AUTO` resolution |
| `CPU_STATIC` + `ONE_MKL` | as above, M2L through grouped SGEMM/DGEMM | explicit; wins on some problem shapes, not all |
| `CUDA_PARTIAL` | M2L and P2P on the device, P2M/M2M/L2L/L2P on the CPU | expansion state crosses at the M2L boundary; supports the potential |
| `CUDA_FULL` | every stage on the device | repeated evaluation transfers moments in and the field out only; field-only |

Unavailable backends raise at construction; the availability queries below
report what this build and machine offer.

In [ ]:
rows = []
rows.append(measure("CPU_STATIC portable", base_options())[0])
if cdfmm.one_mkl_available():
    options = base_options()
    options.static_matrix_backend = cdfmm.StaticMatrixBackend.ONE_MKL
    rows.append(measure("CPU_STATIC oneMKL", options)[0])
if cdfmm.cuda_m2l_p2p_available():
    rows.append(measure("CUDA_PARTIAL", base_options(backend=cdfmm.ExecutionBackend.CUDA_PARTIAL))[0])
if cdfmm.cuda_full_available():
    rows.append(measure("CUDA_FULL", base_options(backend=cdfmm.ExecutionBackend.CUDA_FULL))[0])
print_table(rows, ["case", "precision", "relative L2", "setup s", "eval ms", "p2p packing", "p2m"],
            {"relative L2": ".2e", "setup s": ".3f", "eval ms": ".3f"})

`CPU_REFERENCE` is the independent mathematical traversal: it forms the
Cartesian kernel derivatives of every M2L interaction **during** the
evaluation instead of applying prepared operators, so one evaluation costs
seconds where the static plan costs a millisecond. It exists to validate the
static plans and to teach the algorithm, not to compute with; it is Cartesian
only and accepts point geometry only. One evaluation on a depth-2 tree is
compared with the static plan below.

In [ ]:
reference_options = base_options(order=4, depth=2, precision=cdfmm.StaticPrecision.FLOAT64,
                                 backend=cdfmm.ExecutionBackend.CPU_REFERENCE)
reference_options.expansion_basis = cdfmm.ExpansionBasis.CARTESIAN
static_options = base_options(order=4, depth=2, precision=cdfmm.StaticPrecision.FLOAT64)
static_options.expansion_basis = cdfmm.ExpansionBasis.CARTESIAN
with quiet_construction():
    reference_plan = cdfmm.UniformFmm(positions, positions, reference_options)
    static_plan = cdfmm.UniformFmm(positions, positions, static_options)
start = time.perf_counter()
H_reference = reference_plan.evaluate(moments, target_source_indices=identities)["H"]
reference_seconds = time.perf_counter() - start
start = time.perf_counter()
H_static = static_plan.evaluate(moments, target_source_indices=identities)["H"]
static_seconds = time.perf_counter() - start
print(f"CPU_REFERENCE one evaluation {reference_seconds:.2f} s; CPU_STATIC {static_seconds * 1e3:.2f} ms")
print(f"reference vs static relative L2 difference {field_error_summary(H_reference, H_static)['relative_l2']:.2e}; "
      f"reference vs exact {field_error_summary(H_reference, reference)['relative_l2']:.2e}")

## Near-field (P2P) packings

The exact near-field operator is one canonical set of pair tensors; the
packing is the derived layout an executor streams. `AUTO` applies the measured
policy: point pairs are **recomputed from the positions** (`POINT_GEOMETRY`)
on the CPU and on CUDA FP32, finite bodies keep stored tensors
(`PARTICLE_ROW_SOA` on the CPU, `LEAF_BLOCK` on CUDA), and a `REGULAR_GRID`
layout hint selects the compressed **tensor dictionary** when the built
dictionary is small. Any packing a backend can execute may be forced with
`p2p_packing`; the field does not change.

In [ ]:
rows = []
packings = [
    ("AUTO", cdfmm.P2PExecutionPacking.AUTO),
    ("POINT_GEOMETRY", cdfmm.P2PExecutionPacking.POINT_GEOMETRY),
    ("PARTICLE_ROW_SOA", cdfmm.P2PExecutionPacking.PARTICLE_ROW_SOA),
    ("CANONICAL_AOS", cdfmm.P2PExecutionPacking.CANONICAL_AOS),
    ("TENSOR_DICTIONARY", cdfmm.P2PExecutionPacking.TENSOR_DICTIONARY),
]
for label, packing in packings:
    options = base_options()
    options.p2p_packing = packing
    row, plan = measure(f"p2p_packing={label}", options)
    row["near-field bytes"] = plan.static_plan_statistics["near_field_operator_bytes"]
    rows.append(row)

hinted = base_options()
hinted.spatial_layout = cdfmm.SpatialLayout.REGULAR_GRID
row, plan = measure("spatial_layout=REGULAR_GRID", hinted)
row["near-field bytes"] = plan.static_plan_statistics["near_field_operator_bytes"]
rows.append(row)
print_table(rows, ["case", "p2p packing", "relative L2", "eval ms", "near-field bytes"],
            {"relative L2": ".2e", "eval ms": ".3f"})

# A packing the backend cannot execute is rejected with its reason.
invalid = base_options()
invalid.p2p_packing = cdfmm.P2PExecutionPacking.LEAF_BLOCK
try:
    with quiet_construction():
        cdfmm.UniformFmm(positions, positions, invalid)
except ValueError as error:
    print("\nLEAF_BLOCK on the CPU:", str(error).splitlines()[0])

On CUDA the same request selects among `LEAF_BLOCK`, `CUDA_BSR3`,
`TENSOR_DICTIONARY` (with `cuda_dictionary_target_owned` or
`cuda_dictionary_power2_microtiles` choosing the kernel) and
`POINT_GEOMETRY`. These remain explicit choices for validation and benchmark
studies; the cell is skipped without a device.

In [ ]:
if cdfmm.cuda_full_available():
    rows = []
    for label, packing, flags in [
        ("AUTO", cdfmm.P2PExecutionPacking.AUTO, {}),
        ("LEAF_BLOCK", cdfmm.P2PExecutionPacking.LEAF_BLOCK, {}),
        ("CUDA_BSR3", cdfmm.P2PExecutionPacking.CUDA_BSR3, {}),
        ("TENSOR_DICTIONARY (power-of-two microtiles)", cdfmm.P2PExecutionPacking.TENSOR_DICTIONARY,
         {"cuda_dictionary_power2_microtiles": True}),
        ("POINT_GEOMETRY", cdfmm.P2PExecutionPacking.POINT_GEOMETRY, {}),
    ]:
        options = base_options(backend=cdfmm.ExecutionBackend.CUDA_FULL)
        options.p2p_packing = packing
        for name, value in flags.items():
            setattr(options, name, value)
        row, plan = measure(f"CUDA_FULL {label}", options)
        row["device MiB"] = plan.cuda_plan_statistics["persistent_device_bytes"] / 2**20
        rows.append(row)
    print_table(rows, ["case", "p2p packing", "relative L2", "eval ms", "device MiB"],
                {"relative L2": ".2e", "eval ms": ".3f", "device MiB": ".2f"})
else:
    print("CUDA is not available in this build.")

## Point P2M and L2P: precomputed rows or procedural recurrence

For point sources and point targets the far-field endpoint operators are a
short recurrence of the positions. `PRECOMPUTED` streams coefficient rows
built at construction ($3C$ scalars per point, $C = (p+1)^2$); `PROCEDURAL`
recomputes them during every evaluation and retains only three small factor
tables. The `AUTO` policy is procedural on the CPU hierarchy and on `CUDA_FULL`
FP32, precomputed on `CUDA_FULL` FP64. Finite bodies always keep their exact
precomputed rows.

In [ ]:
rows = []
for label, execution in [
    ("AUTO", cdfmm.PointExpansionExecution.AUTO),
    ("PRECOMPUTED", cdfmm.PointExpansionExecution.PRECOMPUTED),
    ("PROCEDURAL", cdfmm.PointExpansionExecution.PROCEDURAL),
]:
    options = base_options(order=6)
    options.point_expansion_execution = execution
    row, plan = measure(f"point_expansion_execution={label}", options)
    statistics = plan.static_plan_statistics
    row["P2M+L2P bytes"] = statistics["p2m_operator_bytes"] + statistics["l2p_operator_bytes"]
    rows.append(row)
print_table(rows, ["case", "p2m", "relative L2", "eval ms", "P2M+L2P bytes"],
            {"relative L2": ".2e", "eval ms": ".3f"})

## Expansion basis

Both bases share the tree, the near field and the static-plan machinery; they
differ in the far-field operators and the coefficient count. The real
spherical basis is the default because it stores $(p+1)^2$ coefficients
against the Cartesian $(p+1)(p+2)(p+3)/6$ with the same accuracy for point
sources.

In [ ]:
rows = []
for order in (4, 6):
    for label, basis in [("SPHERICAL", cdfmm.ExpansionBasis.SPHERICAL),
                         ("CARTESIAN", cdfmm.ExpansionBasis.CARTESIAN)]:
        options = base_options(order=order, precision=cdfmm.StaticPrecision.FLOAT64)
        options.expansion_basis = basis
        row, plan = measure(f"{label} p={order}", options)
        row["coefficients"] = plan.coefficient_count
        rows.append(row)
print_table(rows, ["case", "coefficients", "relative L2", "eval ms", "host MiB"],
            {"relative L2": ".2e", "eval ms": ".3f", "host MiB": ".2f"})

## Diagnostics

`static_plan_statistics` reports one-time construction cost and retained
bytes per operator, `last_timings` the phases of the most recent evaluation,
and `cuda_plan_statistics` device residency and transfer traffic. These are
the numbers to inspect before tuning anything.

In [ ]:
row, plan = measure("diagnostics", base_options(order=6))
statistics = plan.static_plan_statistics
timings = plan.last_timings

print("construction [s]:")
for key in ("tree_construction_seconds", "universal_operator_build_seconds",
            "p2m_construction_seconds", "l2p_construction_seconds",
            "p2p_construction_seconds", "backend_packing_seconds", "total_setup_seconds"):
    print(f"  {key:36s} {statistics[key]:9.4f}")
print("retained bytes:")
for key in ("m2m_operator_bytes", "m2l_operator_bytes", "l2l_operator_bytes",
            "p2m_operator_bytes", "l2p_operator_bytes", "near_field_operator_bytes",
            "state_bytes", "total_persistent_bytes"):
    print(f"  {key:36s} {statistics[key]:12d}")
print("last evaluation [ms]:")
for key in ("p2m", "m2m", "m2l", "l2l", "l2p", "p2p", "total"):
    print(f"  {key:36s} {timings[key] * 1e3:9.4f}")
if cdfmm.cuda_full_available():
    _, cuda_plan = measure("cuda", base_options(order=6, backend=cdfmm.ExecutionBackend.CUDA_FULL))
    device = cuda_plan.cuda_plan_statistics
    print("CUDA_FULL device:")
    for key in ("persistent_device_bytes", "evaluation_h2d_bytes", "evaluation_d2h_bytes",
                "static_upload_count"):
        print(f"  {key:36s} {device[key]:12d}")

## Recommendations

- Pick the **backend** for the machine (`CPU_STATIC`, or `CUDA_FULL` for
  repeated field evaluation on a GPU) and the **precision** for the accuracy
  you need. Everything else can stay on `AUTO`.
- Set `spatial_layout = REGULAR_GRID` for lattices of identical bodies; it is
  a hint that only costs construction time when it does not apply.
- Force `p2p_packing`, `point_expansion_execution` or the dictionary executor
  only to validate or benchmark an alternative; the result never changes.
- Read `static_plan_statistics` and `last_timings` before tuning $p$ and the
  depth (tutorial 5).